# 10D ring game: DTB and parameter-based transport

With periodic indices $x_0=x_{10}$ and $x_{11}=x_1$, use
$$b_i(x)=x_i-x_i^3-\kappa(2x_i-x_{i-1}-x_{i+1})
          +\omega(x_{i+1}-x_{i-1}).$$
The stochastic model is $dX=b(X)\,dt+\Sigma\,dW$, with ten independent
Brownian motions, $\Sigma=\operatorname{diag}(\sigma_1,\ldots,\sigma_{10})$
and $D=\Sigma\Sigma^\top$. Both methods project the probability-flow velocity
$$v_\rho(x)=b(x)-\tfrac12D\nabla_x\log\rho(x).$$

Run top to bottom. Each method has its own loop and can be rerun independently.
The initial map is exactly the identity; the default cloud is uniform on
$[-2,2]^{10}$ (interpreting the requested two-dimensional box coordinatewise).
Set `STOCHASTIC=False` for the deterministic game. Coupling, rotation, and
noise values below are editable example choices, since none were specified.

The network, parameter utilities, Jacobian, SVD solver, initial-law sampler,
and cloud plots come from
[DTB_Game_Ver2](https://github.com/sun-mengwei/dtb-colab-experiments/tree/codex/game-dynamics-dtb/DTB_Game_Ver2).
The layout follows the
[Cournot example](https://github.com/sun-mengwei/dtb-colab-experiments/blob/codex/game-dynamics-dtb/DTB_Game_Ver2/cournot_3d_nonpotential_stochastic_mlp_dtb.ipynb).


In [24]:
# Shared imports: local checkout, or the existing repository in Colab.
from pathlib import Path
import math
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.func import jacrev, jvp, vmap

required = ('dtb.py', 'run_game_dtb.py', 'utility.py', 'game_visualization.py')
candidates = (Path.cwd(), Path.cwd() / 'DTB_Game_Ver2',
              Path.cwd() / 'dtb-colab-experiments-game-codex' / 'DTB_Game_Ver2')
module_dir = next((p for p in candidates if all((p / f).is_file() for f in required)), None)
if module_dir is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run from the repository root or DTB_Game_Ver2.')
    repo = Path('/content/dtb-colab-experiments')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch',
                        'codex/game-dynamics-dtb',
                        'https://github.com/sun-mengwei/dtb-colab-experiments.git',
                        str(repo)], check=True)
    module_dir = repo / 'DTB_Game_Ver2'
    if not all((module_dir / f).is_file() for f in required):
        raise FileNotFoundError('The Colab checkout must contain DTB_Game_Ver2 utilities.')
sys.path.insert(0, str(module_dir.resolve()))
from dtb import device, flat_params, jform_solve, write_flat_into_model
from run_game_dtb import ResidualMLPMap, game_dtb_basis_matrices, map_at
from utility import sample_initial_with_score
from game_visualization import plot_coordinate_snapshots

print('Reusing utilities from:', module_dir.resolve())


Reusing utilities from: /content/dtb-colab-experiments/DTB_Game_Ver2


## 1. Settings and common initial samples

`H` is the physical time step; one epoch is one time step.
`LOG_EVERY` controls printed progress; every epoch is recorded in `records`.
`RESET_EVERY` controls DTB refits, and `REFIT_EPOCHS` is the optimizer budget
per refit. The parameter method does not refit.

Exact uniform data use the **interior-score heuristic** from the reference
notebook: the singular boundary score is omitted. For a smooth initial
density, choose `INITIAL_LAW='smoothed_uniform'`; this adds Gaussian noise
with standard deviation `4 * SMOOTHING_STD` to the uniform samples and uses
its analytical score. In either case the two methods share the same cloud.


In [25]:
SEED = 2026
DIM, N = 10, 3000
KAPPA, OMEGA = 0.2, 0.5
STOCHASTIC = True
NOISE_STD = (0.15,) * DIM                 # ten independent diffusion amplitudes
INITIAL_LAW = 'uniform'                  # or 'smoothed_uniform'
SMOOTHING_STD = 0.05                     # before scaling [0,1] to [-2,2]
H, FINAL_TIME = 0.02, 2.0
STEPS = round(FINAL_TIME / H)
WIDTH, DEPTH, BASIS_SIZE = 24, 2, 256
SVD_RTOL = 1e-2                          # relative singular-value cutoff
RESET_EVERY, REFIT_EPOCHS, REFIT_LR = 25, 100, 1e-3
LOG_EVERY, CHUNK = 10, 16
COORDINATE_PAIRS = ((1, 2), (1, 6), (5, 10))  # one-based coordinate labels
DEVICE, DTYPE = device(), torch.float64

def game_velocity(x):
    left, right = x.roll(1, dims=-1), x.roll(-1, dims=-1)
    return x - x**3 - KAPPA * (2*x - left - right) + OMEGA * (right - left)

assert DIM == 10 and N > 0 and H > 0 and STEPS > 0
assert min(RESET_EVERY, REFIT_EPOCHS, LOG_EVERY, CHUNK) > 0
assert INITIAL_LAW in ('uniform', 'smoothed_uniform')
assert len(NOISE_STD) == DIM and all(s >= 0 and math.isfinite(s) for s in NOISE_STD)
assert 0 < SVD_RTOL < 1
generator = torch.Generator(device=DEVICE).manual_seed(SEED)
u0, score_u0, _ = sample_initial_with_score(
    N, DIM, law=INITIAL_LAW, device=DEVICE, dtype=DTYPE, generator=generator,
    smoothing_std=SMOOTHING_STD,
)
z, q0 = (4*u0 - 2).detach(), (score_u0 / 4).detach()
sigma = torch.tensor(NOISE_STD, device=DEVICE, dtype=DTYPE)
Sigma = torch.diag(sigma if STOCHASTIC else torch.zeros_like(sigma))
D = Sigma @ Sigma.T
USE_SCORE = bool(torch.any(D != 0))
times = np.arange(STEPS + 1) * H
snapshot_steps = np.unique(np.linspace(0, STEPS, 4, dtype=int))
print(f'{DIM}D, N={N}, T={STEPS*H:g}, device={DEVICE}, diffusion diagonal={D.diag().tolist()}')


10D, N=3000, T=2, device=cuda, diffusion diagonal=[0.0225, 0.0225, 0.0225, 0.0225, 0.0225, 0.0225, 0.0225, 0.0225, 0.0225, 0.0225]


## 2. Small shared helpers

Initialize the existing `ResidualMLPMap` from `run_game_dtb.py` directly:
$T_\theta(z)=z+f_\theta(z)$. Its zero output layer makes $T_{\theta_0}(z)=z$
exactly, so no custom network or frozen copy is needed. Hidden-layer parameter
tangents are initially zero; they can become active after a parameter update
or DTB refit. The SVD handles the resulting rank deficiency. Both methods
use the same seeded initialization and parameter subset.

`dtb.solution_map` also provides the previous anchored formula
$z+f_\theta(z)-f_{\theta_0}(z)$ as a shared function. The residual MLP
already supplies an identity start, so this notebook does not need it.

| Helper | Purpose and reuse |
| :--- | :--- |
| `new_model` | Initialize the imported MLP, call `dtb.flat_params`, and choose the parameter subset $S$. |
| `map_derivatives` | Compute $\partial_zX$ and $\partial_z^2X$. `game_dtb_basis_matrices` instead computes $\partial_{\theta_S}T$. |
| `transported_score` | Recover the current density score from the map derivatives. The related `map_tangent_spatial_terms` provides velocity derivatives for score evolution instead. |
| `tangent_map` | Evaluate $J\alpha$ using the same parameter JVP construction used inside `utility.tangent_velocity_spatial_terms`. |
| `target_velocity` | Assemble this experiment's $b(x)-\frac12Dq(x)$, or just $b(x)$ without diffusion. |
| `relative_error` | Compute the common norm ratio for either method's velocity approximation. |
| `refit` | Fit the existing network on the fixed $(z,X)$ pairs. `fit_map_to_target` draws fresh samples instead. |

For diffusion, track $F=\partial_zX$ and $G=\partial_z^2X$. The score is
$$q(X(z))=F^{-\top}\big(q_0(z)-\nabla_z\log|\det F|\big),\qquad
 [\nabla_z\log|\det F|]_j=\operatorname{tr}(F^{-1}G_{:,:,j}).$$
In the code, `solve(F, G)` forms $F^{-1}G$ without explicitly inverting $F$;
`einsum('niij->nj', ...)` takes its trace for each input coordinate. The
last solve applies $F^{-\top}$ to the corrected reference score.

`utility.tangent_velocity_spatial_terms` and `utility.euler_score_update`
are related, but evolve the score using derivatives in **current positions
$x$**. Here the tangent field is evaluated at **fixed labels $z$**. Replacing
these map derivatives with those utilities directly would omit the coordinate
transformation. The newer `utility.map_tangent_spatial_terms` does apply
that chain rule for an anchored neural map, but returns velocity derivatives
for `euler_score_update`, not the score of the actual discrete map. It also
does not represent the accumulated DTB map across refits. Keeping the helpers
above preserves the same score reconstruction for both methods.

This assumes an invertible transport map. The code stops if a sampled
Jacobian becomes singular or changes orientation; that local check does
not establish global injectivity. No Brownian increments are added to these
probability-flow maps.


In [26]:
def new_model():
    """Initialize the existing MLP, flatten theta, and select tangent coordinates."""
    torch.manual_seed(SEED)
    model = ResidualMLPMap(DIM, WIDTH, DEPTH, activation='tanh',
                           dtype=DTYPE, zero_init_output=True).to(DEVICE)
    theta, structure, _ = flat_params(model)
    if not 1 <= BASIS_SIZE <= theta.numel():
        raise ValueError(f'BASIS_SIZE must be between 1 and {theta.numel()}.')
    rng = torch.Generator().manual_seed(SEED + 1)
    selected = torch.randperm(theta.numel(), generator=rng)[:BASIS_SIZE].sort().values.to(DEVICE)
    return model, theta.detach(), structure, selected


def map_derivatives(batch_map, inputs):
    """Return input Jacobians F:(N,d,d) and Hessians G:(N,d,d,d)."""
    one = lambda point: batch_map(point[None])[0]
    first = jacrev(one)
    second = jacrev(first)
    F, G = [], []
    for chunk in inputs.split(CHUNK):
        F.append(vmap(first)(chunk).detach())
        G.append(vmap(second)(chunk).detach())
    return torch.cat(F), torch.cat(G)


def transported_score(F, G, reference_score):
    """Transform the reference density score using the current map geometry."""
    sign, logdet = torch.linalg.slogdet(F)
    if torch.any(sign <= 0) or not torch.isfinite(logdet).all():
        raise FloatingPointError('Transport Jacobian lost invertibility/orientation; reduce H.')
    n, d, _ = F.shape
    invF_G = torch.linalg.solve(F, G.reshape(n, d, d*d)).reshape(n, d, d, d)
    grad_logdet = torch.einsum('niij->nj', invF_G)
    return torch.linalg.solve(F.transpose(-1, -2),
                              (reference_score - grad_logdet).unsqueeze(-1)).squeeze(-1)


def tangent_map(theta, selected, coefficients, inputs, model, structure):
    """Compute J(inputs) @ coefficients by JVP, without forming the full J."""
    direction = torch.zeros_like(theta).index_copy(0, selected, coefficients)
    return jvp(lambda p: map_at(p, inputs, model, structure), (theta,), (direction,))[1]


def target_velocity(x, F=None, G=None):
    if not USE_SCORE:
        return game_velocity(x).detach()
    q = transported_score(F, G, q0)
    return (game_velocity(x) - 0.5 * (q @ D.T)).detach()


def relative_error(approximation, target):
    return float((approximation - target).norm() / target.norm().clamp_min(1e-30))


def record_epoch(records, epoch, x, alpha, error, start):
    if not torch.isfinite(x).all() or not torch.isfinite(alpha).all() or not math.isfinite(error):
        raise FloatingPointError(f'Non-finite result at epoch {epoch}; reduce H or increase SVD_RTOL.')
    row = dict(epoch=epoch, time=epoch*H, solve_time=(epoch-1)*H,
               relative_error=error, alpha_norm=float(alpha.norm()),
               state_rms=float(x.square().mean().sqrt()), elapsed_seconds=time.perf_counter()-start)
    records.append(row)
    if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == STEPS:
        print(f'epoch {epoch:4d}/{STEPS} | t={epoch*H:.3f} | rel={error:.3e} | '
              f'alpha={row["alpha_norm"]:.3e} | elapsed={row["elapsed_seconds"]:.1f}s')


def refit(model, inputs, targets):
    optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=REFIT_LR)
    for _ in range(REFIT_EPOCHS):
        optimizer.zero_grad(set_to_none=True)
        loss = (model(inputs) - targets).square().mean()
        loss.backward()
        optimizer.step()
    return float((model(inputs).detach() - targets).square().mean().sqrt())


## 3. DTB: update the accumulated map

Within a reset block, freeze $\bar\theta$ and $S$ and evaluate
$J(z)=\partial_{\theta_S}T_{\bar\theta}(z)$ at the fixed samples $z$.
Solve by truncated SVD and update
$$\alpha_k=J^+v_{\rho_k}(X_k(z)),\qquad
  X_{k+1}(z)=X_k(z)+hJ(z)\alpha_k.$$
The error is $\|J\alpha_k-v_k\|/\|v_k\|$.
At a reset, fit the neural map to $(z,X_{k+1}(z))$ only to choose the next
tangent space. **Keep the accumulated state and its derivatives**; refit
error never replaces the solution. The same subset $S$ is retained for
simplicity. `result['map']` evaluates the final accumulated map on new inputs.


In [27]:
def run_dtb():
    model, theta, structure, selected = new_model()
    x = z.clone()                         # X_0(z) = z, independent of refitting
    F = torch.eye(DIM, device=DEVICE, dtype=DTYPE).expand(N, -1, -1).clone()
    G = torch.zeros(N, DIM, DIM, DIM, device=DEVICE, dtype=DTYPE)
    history, records, resets, blocks = [x.cpu().numpy().copy()], [], [], []
    beta = torch.zeros(BASIS_SIZE, device=DEVICE, dtype=DTYPE)
    start = time.perf_counter()
    for k in range(STEPS):
        if k % RESET_EVERY == 0:
            _, _, J = game_dtb_basis_matrices(theta, selected, z, model, structure, chunk=CHUNK)
            J = J.detach()                # frozen throughout this block
        v = target_velocity(x, F, G)
        alpha = jform_solve(J, v.reshape(-1), rtol=SVD_RTOL, method='svd_gpu').detach()
        projected = (J @ alpha).reshape_as(x)
        x_next = (x + H * projected).detach()
        if USE_SCORE:
            velocity_map = lambda inputs: tangent_map(theta, selected, alpha, inputs, model, structure)
            dF, dG = map_derivatives(velocity_map, z)
            F, G = F + H*dF, G + H*dG
            transported_score(F, G, q0)   # also validate the final step
        beta = beta + H * alpha
        record_epoch(records, k+1, x_next, alpha, relative_error(projected, v), start)
        x = x_next
        history.append(x.cpu().numpy().copy())
        if (k+1) % RESET_EVERY == 0 or k+1 == STEPS:
            blocks.append((theta.clone(), beta.clone()))
            beta.zero_()
            if k+1 < STEPS:
                rmse = refit(model, z, x)
                theta = flat_params(model)[0].detach()
                resets.append(dict(epoch=k+1, rmse=rmse))
                print(f'  reset at epoch {k+1}: refit RMSE={rmse:.3e}')

    def final_map(inputs):
        value = inputs.clone()
        for anchor, coefficients in blocks:
            value = value + tangent_map(anchor, selected, coefficients, inputs, model, structure)
        return value.detach()

    return dict(particles=np.stack(history), records=records, resets=resets,
                map=final_map, blocks=blocks, selected=selected, model=model, structure=structure)


In [28]:
dtb_result = run_dtb()


epoch    1/40 | t=0.050 | rel=8.262e-01 | alpha=2.267e+01 | elapsed=5.7s
epoch   10/40 | t=0.500 | rel=9.801e-01 | alpha=5.759e+00 | elapsed=29.6s


FloatingPointError: Transport Jacobian lost invertibility/orientation; reduce H.

## 4. Parameter-based method: update $\theta$

Now $X_k(z)=T_{\theta_k}(z)$. Recompute $J_k(z)$ at every epoch, solve the
same projection problem, and update only the selected coordinates:
$$\theta_{k+1}=\theta_k+hE_S\alpha_k,\qquad
  X_{k+1}(z)=T_{\theta_{k+1}}(z).$$
Measure the **actual neural-map increment**, including nonlinear effects:
$$\varepsilon_k^{\mathrm{rel},T}=
  \frac{\|(T_{\theta_{k+1}}(z)-T_{\theta_k}(z))/h-v_k\|}{\|v_k\|}.$$
The target $v_k$ uses the old state and its score. It is not reevaluated
after updating $\theta$. The score geometry comes from the current neural
map, rather than the tangent Euler step.


In [ ]:
def run_parameter():
    model, theta, structure, selected = new_model()
    x = map_at(theta, z, model, structure).detach()
    F = torch.eye(DIM, device=DEVICE, dtype=DTYPE).expand(N, -1, -1).clone()
    G = torch.zeros(N, DIM, DIM, DIM, device=DEVICE, dtype=DTYPE)
    history, records = [x.cpu().numpy().copy()], []
    start = time.perf_counter()
    for k in range(STEPS):
        v = target_velocity(x, F, G)
        _, _, J = game_dtb_basis_matrices(theta, selected, z, model, structure, chunk=CHUNK)
        alpha = jform_solve(J.detach(), v.reshape(-1), rtol=SVD_RTOL, method='svd_gpu').detach()
        theta_next = theta.clone()
        theta_next[selected] += H * alpha
        x_next = map_at(theta_next, z, model, structure).detach()
        finite_difference = (x_next - x) / H
        error = relative_error(finite_difference, v)
        if USE_SCORE:
            next_map = lambda inputs: map_at(theta_next, inputs, model, structure)
            F, G = map_derivatives(next_map, z)
            transported_score(F, G, q0)
        record_epoch(records, k+1, x_next, alpha, error, start)
        theta, x = theta_next, x_next
        history.append(x.cpu().numpy().copy())
    write_flat_into_model(model, theta, structure)
    return dict(particles=np.stack(history), records=records, theta=theta,
                map=lambda inputs: map_at(theta, inputs, model, structure).detach(),
                selected=selected, model=model, structure=structure)


In [ ]:
parameter_result = run_parameter()


## 5. Diagnostics and coordinate clouds

Relative errors are empirical $L^2$ ratios over the same fixed reference
samples (the common $1/\sqrt{N}$ cancels). These measure one-step velocity
agreement, not error against an exact stochastic solution. DTB uses the
projection residual; the parameter method uses the map finite difference.
Cloud plots share axes across methods and time. The run dictionaries retain
all epoch records and particle states; `map(z_new)` evaluates either result.


In [ ]:
runs = {'DTB': dtb_result, 'Parameter': parameter_result}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for name, result in runs.items():
    records = result['records']
    t = [r['solve_time'] for r in records]
    error_label = 'DTB: projection' if name == 'DTB' else 'Parameter: map difference / h'
    axes[0].plot(t, [r['relative_error'] for r in records], label=error_label)
    axes[1].plot(t, [r['alpha_norm'] for r in records], label=name)
for ax, title, ylabel in zip(axes, ['Relative velocity error', 'Coefficient norm'],
                             [r'$\varepsilon_k^{\mathrm{rel}}$', r'$\|\alpha_k\|_2$']):
    ax.set(title=title, xlabel='Time at projection', ylabel=ylabel)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.show()

cloud_figures = {}
for name, result in runs.items():
    fig = plot_coordinate_snapshots(result['particles'], times, snapshot_steps,
                                   coordinate_pairs=COORDINATE_PAIRS, trail_particles=0)
    fig.suptitle(f'{name}: coordinate clouds')
    axes = np.asarray(fig.axes).reshape(len(COORDINATE_PAIRS), len(snapshot_steps))
    for row, pair in enumerate(COORDINATE_PAIRS):
        limits = []
        for coordinate in pair:
            values = np.concatenate([r['particles'][:, :, coordinate-1].ravel() for r in runs.values()])
            pad = 0.05 * max(float(np.ptp(values)), 1e-3)
            limits.append((float(values.min()) - pad, float(values.max()) + pad))
        for ax in axes[row]:
            ax.set(xlim=limits[0], ylim=limits[1])
    cloud_figures[name] = fig
plt.show()

for name, result in runs.items():
    last = result['records'][-1]
    print(f'{name}: {last["epoch"]} epochs, final relative error={last["relative_error"]:.3e}, '
          f'alpha norm={last["alpha_norm"]:.3e}')
